## Ultrasonic Audio Steganography Demo

This notebook demonstrates how to use the package to hide data inside the ultrasonic frequencies of a `.wav` file. We will explore two primary features:
1. **Spectrogram Embedding**: Visualizing an image inside an audio file's spectrogram.
2. **Data Hiding**: Reversibly encoding and decoding raw binary data (like an image file) into the ultrasonic spectrum.

### Setup and Environment Preparation

First, we will import the required libraries and generate mock source assets (a base audio file and a dummy image) so this notebook can run independently without external file dependencies.

In [ ]:
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from PIL import Image
from scipy.signal import spectrogram

# Import functions from package file
from ultrasonic_img_steg import encode_spectrogram, encode_data, decode_data

# Create demo image, a checkerboard pattern
block_grid = np.indices((8, 16)).sum(axis=0) % 2
pixel_array = block_grid.repeat(16, axis=0).repeat(16, axis=1)
pixel_array = (pixel_array * 255).astype(np.uint8)
img = Image.fromarray(pixel_array, mode="L")
img.save("image.png")

# Generate a mock base audio file (44.1 kHz, mono, 5 seconds of sound, to
# make the spectrogram image visible without zooming in)
sample_rate = 44100
duration = 7.0
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
base_audio = 0.1 * np.sin(2 * np.pi * 440 * t) + 0.001 * np.random.randn(len(t)) # 440Hz tone + noise
sf.write("input.wav", base_audio, sample_rate)

print("Setup complete. 'image.png' and 'input.wav' generated successfully.")

### Embed Image into Spectrogram

This function draws an image directly into the magnitude array of the audio's Short-Time Fourier Transform (STFT) between 19 kHz and 22 kHz. When you plot the spectrogram of the resulting output file, the shape becomes visible.

In [ ]:
# Execute the spectrogram encoding
encode_spectrogram(
    audio_path="input.wav",
    image_path="image.png",
    output_path="output_spectrogram.wav",
    start_freq=19000,
    end_freq=22000,
    fft_size=4096,
    hop=512,
    strength=0.0003   # The strength determines the brightness of the embedded image
)
print("Spectrogram encoding complete.")

#### Verify the Spectrogram Visual

We will compute and plot the spectrogram of the newly generated `output_spectrogram.wav` file to reveal the hidden visual pattern.

In [ ]:
# Load the output audio
audio_signal, sr = sf.read("output_spectrogram.wav")

# Compute the spectrogram
frequencies, times, spectrogram_matrix = spectrogram(audio_signal, fs=sr, nperseg=2048)

# Plot the results
plt.figure(figsize=(10, 6))
plt.pcolormesh(times, frequencies, 10 * np.log10(spectrogram_matrix + 1e-10), shading='gouraud', cmap='magma')
plt.title("Audio Spectrogram Analysis")
plt.ylabel("Frequency [Hz]")
plt.xlabel("Time [sec]")
plt.ylim(19000, 22000)
plt.colorbar(label="Power/Frequency [dB]")
plt.show()

### Encode and Decode Raw Binary Data

Unlike the first feature, this function transforms raw binary file data directly into complex frequency symbols via Fast Fourier Transforms (FFT). This allows perfect structural reconstruction of files.

In [ ]:
# Step A: Encode the raw PNG file bytes into the audio frequencies
encode_data(
    audio_path="input.wav",
    image_path="image.png",
    output_path="output_data.wav",
    start_freq=19000,
    end_freq=22000
)
print("Data injection into audio complete.")

In [ ]:
# Step B: Extract the raw file bytes back out of the audio
decode_data(
    encoded_audio_path="output_data.wav",
    output_image_path="decoded_image.png",
    start_freq=19000,
    end_freq=22000
)
print("Data extraction complete.")

#### Verify Data Integrity

Let's display the original image side-by-side with the completely extracted image file to confirm the process did not degrade or drop data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

# Load original
orig_img = Image.open("image.png")
axes[0].imshow(orig_img, cmap='gray')
axes[0].set_title("Original Source Image")
axes[0].axis('off')

# Load recovered
rec_img = Image.open("decoded_image.png")
axes[1].imshow(rec_img, cmap='gray')
axes[1].set_title("Extracted Image File")
axes[1].axis('off')

plt.show()

# Exact byte comparison verification
bytes_original = open("image.png", "rb").read()
bytes_recovered = open("decoded_image.png", "rb").read()
print(f"Data Match Verification: {bytes_original == bytes_recovered}")